# 05 - Model Training (3-Layer Pipeline)

Loads the Layer 1 selector and final feature list produced by notebook 04, re-applies Layer 2 feature engineering to the full training set, filters to the final feature list, tunes XGBoost with Optuna, trains the final model, and saves the model artifact.

In [3]:
# ! python -m pip install optuna

In [4]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import joblib
import json
import config
from src.io import logger
from src.models import build_model, xgb_safe_frame
from src.optimization import optimize_model
from src.feature_selection import create_extended_engineered_features
from sklearn.metrics import roc_auc_score

d:\Prostate_BCR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Preprocessed Training Data

In [5]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)

y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df["BCR"]

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

2026-09-01 02:02:17 | INFO     | prostate_bcr | Training data loaded: (343, 19018), Positives: 46


## Step 2: Load Artifacts From Notebook 04

Requires `fitted_layer1_selector.joblib` and `selected_features_final.csv`. Raises a clear error if notebook 04 has not been run.

In [6]:
selector_path = config.MODELS_DIR / "fitted_layer1_selector.joblib"
features_path = config.TABLES_DIR / "selected_features_final.csv"

try:
    fitted_l1 = joblib.load(selector_path)
    selected_df = pd.read_csv(features_path)
    final_features = selected_df["feature"].tolist()
    logger.info(f"Loaded {len(final_features)} final features from the 3-layer pipeline.")
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"{e}\n\n"
        "Please run '04_feature_selection.ipynb' first to generate "
        "'fitted_layer1_selector.joblib' and 'selected_features_final.csv'."
    )

2026-09-01 02:02:17 | INFO     | prostate_bcr | Loaded 30 final features from the 3-layer pipeline.


## Step 3: Apply Layer 2 Feature Engineering and Filter to Final Features

Engineering is applied to the full training set before filtering, matching how notebook 04 built the candidate pool. Any final feature missing after engineering (should not normally happen, but can for edge splits) is filled with 0.0.

In [7]:
X_train_eng, _ = create_extended_engineered_features(
    X_train, selected_genes=fitted_l1["mi_features"]
)

available_in_train = [f for f in final_features if f in X_train_eng.columns]
X_train_final = X_train_eng[available_in_train].copy()

missing_feats = set(final_features) - set(available_in_train)
if missing_feats:
    logger.warning(f"Missing {len(missing_feats)} features in training data: {list(missing_feats)[:5]}...")
    for feat in missing_feats:
        X_train_final[feat] = 0.0
    X_train_final = X_train_final[final_features]

logger.info(f"Final training matrix shape: {X_train_final.shape}")

2026-09-01 02:02:17 | INFO     | prostate_bcr | Layer 2 - Created 4 clean engineered features
2026-09-01 02:02:17 | INFO     | prostate_bcr | Final training matrix shape: (343, 30)


## Step 4: Hyperparameter Tuning with Optuna

In [8]:
logger.info("Starting Optuna hyperparameter tuning for XGBoost...")
n_trials = config.N_OPTUNA_TRIALS if hasattr(config, "N_OPTUNA_TRIALS") else 100
_, study, best_params = optimize_model(
    model_name="XGBoost",
    X_train=X_train_final,
    y_train=y_train,
    n_trials=n_trials,
    cv_splits=config.INNER_SPLITS,
    scoring="roc_auc",
)

logger.info(f"Best CV AUC: {study.best_value:.4f}")
logger.info(f"Best params: {json.dumps(best_params, indent=2, default=str)}")

pd.DataFrame([best_params]).to_csv(config.TABLES_DIR / "best_hyperparameters.csv", index=False)

2026-09-01 02:02:17 | INFO     | prostate_bcr | Starting Optuna hyperparameter tuning for XGBoost...
d:\Prostate_BCR\venv\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-09-01 02:02:17,454] A new study created in memory with name: XGBoost_BCR_Optimization
2026-09-01 02:02:17 | INFO     | prostate_bcr | Created Optuna study: XGBoost_BCR_Optimization
2026-09-01 02:02:17 | INFO     | prostate_bcr | Starting Optuna optimization for XGBoost: 100 trials
Best trial: 0. Best value: 0.844753:   1%|          | 1/100 [00:02<03:40,  2.23s/it]

[I 2026-09-01 02:02:19,684] Trial 0 finished with value: 0.844753086419753 and parameters: {'max_depth': 5, 'min_child_weight': 6.351221010640703, 'gamma': 0.007177141927992002, 'learning_rate': 0.030405325392865647, 'n_estimators': 200, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'colsample_bylevel': 0.9464704583099741, 'reg_alpha': 0.002570603566117598, 'reg_lambda': 0.023585940584142682}. Best is trial 0 with value: 0.844753086419753.


Best trial: 0. Best value: 0.844753:   2%|▏         | 2/100 [00:03<03:03,  1.87s/it]

[I 2026-09-01 02:02:21,301] Trial 1 finished with value: 0.7993265993265993 and parameters: {'max_depth': 3, 'min_child_weight': 7.579479953348009, 'gamma': 0.04566054873446119, 'learning_rate': 0.0033572967053517922, 'n_estimators': 250, 'subsample': 0.6733618039413735, 'colsample_bytree': 0.7216968971838151, 'colsample_bylevel': 0.8099025726528951, 'reg_alpha': 7.71800699380605e-05, 'reg_lambda': 4.17890272377219e-06}. Best is trial 0 with value: 0.844753086419753.


Best trial: 0. Best value: 0.844753:   3%|▎         | 3/100 [00:06<03:21,  2.07s/it]

...

Best trial: 78. Best value: 0.896423:  99%|█████████▉| 99/100 [00:46<00:00,  3.69it/s]

[I 2026-09-01 02:03:03,670] Trial 98 finished with value: 0.8847081930415265 and parameters: {'max_depth': 3, 'min_child_weight': 0.0027130451337387773, 'gamma': 1.8658841488076528e-08, 'learning_rate': 0.06344462360144637, 'n_estimators': 900, 'subsample': 0.7551883937480078, 'colsample_bytree': 0.6319027901563588, 'colsample_bylevel': 0.7952229737238907, 'reg_alpha': 0.00032216148856962373, 'reg_lambda': 0.0014780534420201596}. Best is trial 78 with value: 0.8964225589225588.


Best trial: 78. Best value: 0.896423: 100%|██████████| 100/100 [00:46<00:00,  2.15it/s]
2026-09-01 02:03:04 | INFO     | prostate_bcr | Optimization complete! Best roc_auc = 0.8964


[I 2026-09-01 02:03:04,051] Trial 99 finished with value: 0.8781004489337821 and parameters: {'max_depth': 5, 'min_child_weight': 0.002786969965609172, 'gamma': 8.866205904096556e-08, 'learning_rate': 0.04768847739775775, 'n_estimators': 700, 'subsample': 0.7405127261018988, 'colsample_bytree': 0.6510600944248816, 'colsample_bylevel': 0.6962057167912628, 'reg_alpha': 0.00022333516779499246, 'reg_lambda': 5.545158801369788e-05}. Best is trial 78 with value: 0.8964225589225588.


2026-09-01 02:03:04 | INFO     | prostate_bcr | Best CV AUC: 0.8964
2026-09-01 02:03:04 | INFO     | prostate_bcr | Best params: {
  "max_depth": 5,
  "min_child_weight": 0.00121293134156988,
  "gamma": 3.221105111559504e-06,
  "learning_rate": 0.15762225012199652,
  "n_estimators": 800,
  "subsample": 0.6250357305691686,
  "colsample_bytree": 0.610327867840577,
  "colsample_bylevel": 0.7429305716548712,
  "reg_alpha": 4.6526698427305736e-06,
  "reg_lambda": 0.0002666419893873968,
  "scale_pos_weight": 6.456521739130435,
  "random_state": 42,
  "n_jobs": -1,
  "eval_metric": "logloss"
}


## Step 5: Train Final Model on Full Training Set

In [9]:
# Cell for Training Final Model in Notebook 05
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
best_params["scale_pos_weight"] = n_neg / max(n_pos, 1)
best_params["random_state"] = config.RANDOM_STATE
best_params["n_jobs"] = -1
best_params["eval_metric"] = "logloss"

# FIX: Pass y_train explicitly to build_model
final_model = build_model("XGBoost", y_train=y_train, **best_params)

X_train_safe = xgb_safe_frame(X_train_final)
final_model.fit(X_train_safe, y_train)

# Sanity Check
train_pred = final_model.predict_proba(X_train_safe)[:, 1]
train_auc = roc_auc_score(y_train, train_pred)
logger.info(f"Training AUC (sanity check): {train_auc:.4f}")

# Save Artifacts IMMEDIATELY after training
joblib.dump(final_model, config.MODELS_DIR / "best_model_xgboost.joblib")
print("✅ Model trained and saved successfully.")

2026-09-01 02:03:04 | INFO     | prostate_bcr | Built model: XGBoost
2026-09-01 02:03:04 | INFO     | prostate_bcr | Training AUC (sanity check): 1.0000


✅ Model trained and saved successfully.


## Step 6: Save Artifacts

In [10]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, config.MODELS_DIR / "best_model_xgboost.joblib")
joblib.dump(fitted_l1, config.MODELS_DIR / "fitted_layer1_selector.joblib")

print("Model training complete. Artifacts saved:")
print(f"  - {config.MODELS_DIR / 'best_model_xgboost.joblib'}")
print(f"  - {config.MODELS_DIR / 'fitted_layer1_selector.joblib'}")

Model training complete. Artifacts saved:
  - D:\Prostate_BCR\core\outputs\models\best_model_xgboost.joblib
  - D:\Prostate_BCR\core\outputs\models\fitted_layer1_selector.joblib
